# 08 — Gold Facts & KPI Tables

**Purpose:** Build Gold fact tables at their correct event grains and create source-supported KPI aggregates.

Fact grains:
- `fact_card_transactions`: one row per `transaction_id`.
- `fact_authentication_events`: one row per `auth_event_id`.
- `fact_payment_status_changes`: one row per `change_event_id`.

The three event streams remain separate. Small customer, merchant and card dimensions are broadcast only when joining to the transaction fact. KPI classifications must be confirmed from the delivered values and business-rule matrix; this notebook does not guess status meanings.


In [0]:
from pyspark.sql.functions import col, broadcast, count

SILVER = "capstone_team01.silver_schema"
GOLD = "capstone_team01.gold_schema"

spark.sql("CREATE SCHEMA IF NOT EXISTS capstone_team01.gold_schema")


## 1. Read Silver tables and Gold dimensions


In [0]:
transactions = spark.table(f"{SILVER}.silver_card_transactions")
auth_events = spark.table(f"{SILVER}.silver_device_authentication_events")
status_changes = spark.table(f"{SILVER}.silver_payment_status_changes")

customer = spark.table(f"{GOLD}.dim_customer")
merchant = spark.table(f"{GOLD}.dim_merchant")
card = spark.table(f"{GOLD}.dim_card")

print("Transactions:", transactions.count())
print("Authentication events:", auth_events.count())
print("Payment status changes:", status_changes.count())


## 2. Fact Card Transactions

Join the transaction fact to the small dimensions using broadcast. Keep the existing business IDs; do not add surrogate keys.


In [0]:
fact_card_transactions = (
    transactions.alias("t")
    .join(broadcast(customer.alias("c")),
          col("t.customer_id") == col("c.customer_id"), "left")
    .join(broadcast(merchant.alias("m")),
          col("t.merchant_id") == col("m.merchant_id"), "left")
    .join(broadcast(card.alias("d")),
          col("t.card_id") == col("d.card_id"), "left")
    .select(
        "t.*",
        col("c.customer_segment"),
        col("c.risk_band").alias("customer_risk_band"),
        col("m.merchant_name"),
        col("m.merchant_category"),
        col("m.merchant_country"),
        col("m.risk_band").alias("merchant_risk_band"),
        col("d.card_type"),
        col("d.card_status"),
        col("d.network")
    )
)

fact_card_transactions.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{GOLD}.fact_card_transactions")

print("Fact card transactions:", fact_card_transactions.count())
display(fact_card_transactions.limit(5))


## 3. Fact Authentication Events

Keep authentication events at their own event grain. Do not join them directly to payment status events.


In [0]:
fact_authentication_events = auth_events.select(
    "auth_event_id",
    "transaction_id",
    "customer_id",
    "event_ts",
    "device_id",
    "authentication_method",
    "authentication_result",
    "ip_country",
    "attempt_number",
    "risk_signal"
)

fact_authentication_events.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{GOLD}.fact_authentication_events")

print("Fact authentication events:", fact_authentication_events.count())
display(fact_authentication_events.limit(5))


## 4. Fact Payment Status Changes

Keep every status change event. `transaction_id` is not the event key; use `change_event_id`.


In [0]:
fact_payment_status_changes = status_changes.select(
    "change_event_id",
    "transaction_id",
    "change_ts",
    "old_status",
    "new_status",
    "reason_code",
    "change_sequence",
    "source_system",
    "operator_id",
    "correlation_id"
)

fact_payment_status_changes.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{GOLD}.fact_payment_status_changes")

print("Fact payment status changes:", fact_payment_status_changes.count())
display(fact_payment_status_changes.limit(5))


## 5. Profile business values before KPI classification

Review the actual values. Do not assume which payment status means authorized/failed or which risk signal means suspicious. Confirm the mappings against the approved business-rule matrix.


In [0]:
print("Transaction payment_status values")
display(fact_card_transactions.groupBy("payment_status").count().orderBy("payment_status"))

print("Payment status change values")
display(fact_payment_status_changes.groupBy("new_status").count().orderBy("new_status"))

print("Authentication result values")
display(fact_authentication_events.groupBy("authentication_result").count().orderBy("authentication_result"))

print("Authentication risk_signal values")
display(fact_authentication_events.groupBy("risk_signal").count().orderBy("risk_signal"))


## 6. Transaction volume by merchant and payment status

This aggregate is directly supported by the project specification and does not require classifying statuses as success or failure.


In [0]:
gold_merchant_payment_volume = (
    fact_card_transactions
    .groupBy("merchant_id", "merchant_name", "payment_status")
    .agg(count("transaction_id").alias("transaction_count"))
)

gold_merchant_payment_volume.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{GOLD}.gold_merchant_payment_volume")

display(gold_merchant_payment_volume.orderBy(col("transaction_count").desc()).limit(20))


## 7. Authentication pattern aggregate

Summarize authentication events by method, result and risk signal. This exposes patterns without inventing a suspicious-value classification.


In [0]:
gold_authentication_patterns = (
    fact_authentication_events
    .groupBy("authentication_method", "authentication_result", "risk_signal")
    .agg(
        count("auth_event_id").alias("authentication_event_count"),
        count("transaction_id").alias("transaction_reference_count")
    )
)

gold_authentication_patterns.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{GOLD}.gold_authentication_patterns")

display(gold_authentication_patterns.orderBy(col("authentication_event_count").desc()))


## 8. KPI classification note

The v5 specification requires:
- Daily authorized value
- Merchant failure rate
- Suspicious transaction rate

The delivered fields contain payment statuses and authentication risk signals, but their business meanings must be confirmed before classifying records. After reviewing the profiles above and confirming the approved mappings, implement these three KPI calculations with documented numerator, denominator, date logic and null handling. Do not publish rates using guessed status or risk values.


In [0]:
# Final Gold table counts
print("fact_card_transactions:", spark.table(f"{GOLD}.fact_card_transactions").count())
print("fact_authentication_events:", spark.table(f"{GOLD}.fact_authentication_events").count())
print("fact_payment_status_changes:", spark.table(f"{GOLD}.fact_payment_status_changes").count())
print("gold_merchant_payment_volume:", spark.table(f"{GOLD}.gold_merchant_payment_volume").count())
print("gold_authentication_patterns:", spark.table(f"{GOLD}.gold_authentication_patterns").count())
